In [1]:
import numpy as np
import pandas as pd
from scipy.io import mmread

## P2 (a)

In [2]:
# Load the matrix
# bcspwr09     (Power Networks)

A = mmread("/Users/lpwer/Documents/NetSIPhD/PHYS7332_NetData/HW2/power-bcspwr09/power-bcspwr09.mtx").tocoo()

# Save as edge list
with open("edges.csv", "w") as f:
    f.write("Source,Target\n")
    for i, j in zip(A.row, A.col):
        if i != j:  # optional: skip self-loops
            f.write(f"{i},{j}\n")

Modularity: 0.893

Modularity with resolution: 0.893

Number of Communities: 24

In [3]:
df = pd.read_csv('/Users/lpwer/Documents/NetSIPhD/PHYS7332_NetData/HW2/filtered_table.csv')

partition_MM = dict(zip(df['Id'], df['modularity_class']))

## P2 (b)

In [2]:
from graph_tool.all import *
import graph_tool.all as gt
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from matplotlib import rc
rc('axes', fc='w')
rc('figure', fc='w')
rc('savefig', fc='w')
rc('axes', axisbelow=True)
import json

In [7]:
df_edges = pd.read_csv('/Users/lpwer/Documents/NetSIPhD/PHYS7332_NetData/phys7332_fa25/HW2/edges.csv')
# x = pd.concat([df_edges[ucal], df_edges[vcal]])
# print(x)
labels = pd.Index(pd.unique(pd.concat([df_edges['Source'], df_edges['Target']])))
# print(labels)

label_to_int = {lab: i for i, lab in enumerate(labels)}
# print(labels_to_int)

df_int = df_edges.copy()
df_int['Source'] = df_int['Source'].map(label_to_int)
df_int['Target'] = df_int['Target'].map(label_to_int)

print(df_int)

      Source  Target
0          0     966
1          1     967
2          2     968
3          3     969
4          4     970
...      ...     ...
4783     367     963
4784      42     257
4785     257     964
4786     124     965
4787     575      52

[4788 rows x 2 columns]


In [17]:
g = gt.Graph(directed=False)
g.add_vertex(len(labels))
g.add_edge_list(df_int[['Source', 'Target']].itertuples(index=False, name=None))
gt.remove_self_loops(g)
gt.remove_parallel_edges(g)
pos = gt.sfdp_layout(g)
g.vp.pos = pos

# 1) recreat mapping
int_to_label = {i: lab for lab, i in label_to_int.items()}

name = g.new_vertex_property("string")
for i in range(g.num_vertices()):
    name[g.vertex(i)] = str(int_to_label[i])
g.vp['name'] = name            

# 2) Save positions once and keep them on THIS 'g'
pos = gt.sfdp_layout(g)
g.vp['pos'] = pos

# 3) Community detection (SBM)
state = gt.minimize_blockmodel_dl(g)
blocks = state.get_blocks()
B = state.get_B()

# 4) Colors
cmap = matplotlib.colormaps.get_cmap("tab20")  # continuous colormap
palette = [cmap(i / max(B - 1, 1))[:3] for i in range(B)]
vcolor = g.new_vertex_property("vector<double>")
for v in g.vertices():
    vcolor[v] = palette[int(blocks[v])]

gt.graph_draw(
    g, pos=g.vp['pos'],
    vertex_fill_color=vcolor,
    vertex_size=5, edge_pen_width=0.5,
    output="network_colored_by_community.png"
)

# 5) Export partition
partition_gt = {g.vp['name'][v]: int(blocks[v]) for v in g.vertices()}

import json
with open("partition.json", "w") as f:
    json.dump(partition, f, indent=2)

g.save("network_with_pos_and_name.gt.gz")

In [18]:
print(partition_gt)

{'1416': 761, '1174': 241, '1106': 969, '1083': 761, '598': 29, '1504': 623, '528': 241, '8': 29, '90': 29, '10': 623, '484': 623, '13': 761, '284': 761, '597': 29, '1641': 623, '1648': 1055, '20': 1055, '596': 1055, '1581': 245, '1568': 761, '595': 1085, '26': 761, '27': 761, '594': 761, '593': 1085, '31': 29, '592': 29, '33': 29, '591': 29, '590': 1085, '1173': 623, '1153': 241, '488': 241, '1410': 969, '521': 241, '1172': 1055, '42': 623, '589': 623, '588': 1055, '46': 1085, '1121': 1085, '587': 245, '1681': 761, '586': 1055, '1650': 761, '1624': 623, '1521': 1085, '585': 1085, '584': 1085, '58': 1085, '1658': 623, '967': 1085, '1722': 623, '1605': 226, '507': 29, '1171': 241, '1491': 29, '1591': 623, '69': 1055, '70': 1055, '1508': 1055, '72': 1055, '486': 1055, '1168': 623, '1097': 241, '1167': 1085, '1459': 1085, '583': 1085, '1627': 1055, '1166': 241, '1091': 1055, '85': 1055, '1143': 29, '88': 29, '89': 29, '706': 29, '1540': 969, '582': 1085, '1642': 623, '1161': 623, '96': 62